In [62]:
import pandas as pd

df_marking = pd.read_csv('df_with_marking_final.csv')

In [63]:
df_marking["precrime_argument"].value_counts()

precrime_argument
была    92
нет      8
Name: count, dtype: int64

In [67]:
import re

def get_full_text(row):
    return f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"

precrime_argument_patterns = [
    r"\bссора\b",
    r"\bссор[аиые]\b",
    r"\bконфликт[а-я]*\b",
    r"\bдрака\b",
    r"\bоскорблял[аие]\b",
    r"\bв результате (личной|взаимной) неприязни\b",
    r"\bна почве (возникшей )?(личной|взаимной)? ?неприязни\b",
    r"\bна почве ссоры\b",
    r"\b(вызванной|вызванного) .*? ссор[аы]\b",
    r"\bпрепирательств[ао]?\b",
    r"\bвыяснени[ея] отношений\b"
]
noprecrime_argument_patterns = [
    r"\bссоры (не (предшествовал[аи]?|было|возникало|наблюдалось))\b",
    r"\bконфликта не (было|возникло)\b",
    r"\b(действовал|убил) внезапно\b",
    r"\b(внезапно|импульсивно) возник(ший|шее|шееся) намерение\b",
    r"\bбез (предшествующей|какой-либо) ссоры\b",
    r"\bвнезапно возникшее чувство\b",
    r"\bбез выяснения отношений\b",
    r"\bне ругались, не ссорились\b",
    r"\bсловесн[а-я]* перепалки не было\b",
    r"\bникаких конфликтов между ними не было\b"
]

train_data_precrime_argument = []
not_found = 0
skipped = 0

for idx, row in df_marking.iterrows():
    true_label = str(row.get("precrime_argument")).strip().lower()
    if true_label not in ["была", "нет"]:
        skipped += 1
        continue

    text = get_full_text(row)
    text_lower = text.lower()

    found = False

    if true_label == "была":
        for pattern in precrime_argument_patterns:
            match = re.search(pattern, text_lower)
            if match:
                start, end = match.span()
                train_data_precrime_argument.append((text, {"entities": [(start, end, "PRECRIME_ARGUMENT")]}))
                # print(text[start:end])
                found = True
                break
        if not found:
            train_data_precrime_argument.append((text, {"entities": []}))
            not_found += 1

    elif true_label == "нет":
        # Добавим только пустую разметку
        train_data_precrime_argument.append((text, {"entities": []}))

print(f"TRAIN_DATA_PRECRIME_ARGUMENT готово: {len(train_data_precrime_argument)} примеров")

TRAIN_DATA_PRECRIME_ARGUMENT готово: 100 примеров


In [ ]:
print(train_data_precrime_argument[1][1])

{'entities': [(1393, 1398, 'PRECRIME_ARGUMENT')]}


In [68]:
import spacy
from spacy.training.example import Example
from spacy.util import minibatch
from tqdm import tqdm

nlp = spacy.blank("ru")
ner = nlp.add_pipe("ner")

ner.add_label("PRECRIME_ARGUMENT")

optimizer = nlp.begin_training()

for itn in range(15):  # количество эпох
    losses = {}
    examples = [Example.from_dict(nlp.make_doc(text), annotations) for text, annotations in train_data_precrime_argument]
    for batch in minibatch(examples, size=16):
        nlp.update(batch, drop=0.3, losses=losses)
    print(f"Epoch {itn + 1}, Loss: {losses['ner']:.4f}")

Epoch 1, Loss: 500854.3119
Epoch 2, Loss: 37890.2866
Epoch 3, Loss: 184.0011
Epoch 4, Loss: 187.8942
Epoch 5, Loss: 184.2050
Epoch 6, Loss: 170.8891
Epoch 7, Loss: 3820.1039
Epoch 8, Loss: 543.7763
Epoch 9, Loss: 146.9424
Epoch 10, Loss: 135.3257
Epoch 11, Loss: 142.6632
Epoch 12, Loss: 124.8200
Epoch 13, Loss: 126.8529
Epoch 14, Loss: 125.0923
Epoch 15, Loss: 122.1841


In [69]:
from sklearn.metrics import accuracy_score, f1_score

y_true = []
y_pred = []

for text, annotation in train_data_precrime_argument:
    doc = nlp(text)
    has_prediction = any(ent.label_ == "PRECRIME_ARGUMENT" for ent in doc.ents)
    predicted_label = "была" if has_prediction else "нет"
    true_label = "была" if annotation["entities"] else "нет"
    y_true.append(true_label)
    y_pred.append(predicted_label)

accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted') 
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.3f}")

Accuracy: 0.5800
F1 Score: 0.670
